In [ ]:
pip install ollama

# Installing Ollama Framework

In [ ]:
import ollama

# Importing Ollama Framework

In [2]:
import subprocess

result = subprocess.run(
    [r"C:\Users\ASUS\AppData\Local\Programs\Ollama\ollama.exe", "list"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

NAME           ID              SIZE      MODIFIED       
llama3.2:3b    a80c4f17acd5    2.0 GB    39 minutes ago    




subprocess.run(...) executes an external program.

r"..." creates a raw string, so backslashes in the Windows path are handled correctly.

"list" is the Ollama command that shows all installed models.

capture_output=True captures both standard output and error output.

text=True returns the output as strings instead of bytes.

result.stdout contains the list of installed models.

result.stderr contains any error messages.

In [ ]:
import subprocess

subprocess.run([
    r"C:\Users\ASUS\AppData\Local\Programs\Ollama\ollama.exe",
    "pull",
    "llama3.2:3b"
])

# This code will download (pull) the Llama 3.2 3B model into Ollama

CompletedProcess(args=['C:\\Users\\ASUS\\AppData\\Local\\Programs\\Ollama\\ollama.exe', 'pull', 'llama3.2:3b'], returncode=0)

In [ ]:
response = ollama.chat(
    model= "llama3.2:3b",
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)
 
print(response["message"]["content"])

# the Ollama Python library to chat with the local llama3.2:3b model
# ollama.chat() sends a prompt to the specified model.
# model="llama3.2:3b" selects the local model.
# messages follows the chat format with roles such as user, assistant, and system.
# The model's reply is returned in response["message"]["content"].

Hello!


"role": "system" --------->   Tells the AI these are instructions and The system message controls the AI's behavior.

      Tells the AI that this message contains instructions.

      The system message controls the AI's behavior, role, tone, and rules.

"content": SYSTEM_PROMPT------->   Uses the value stored in SYSTEM_PROMPT.

        Uses the value stored in the variable SYSTEM_PROMPT.

        This is the actual instruction given to the AI.

"role": "user"--------->   Indicates that this message comes from the user.

"content": user_message --------> Parameter for ask function and message is the content you are asking the system

        Uses the value passed to the ask() function.

        Contains the question or request that the user wants the AI to answer.

In [ ]:
SYSTEM_PROMPT = """
You are Aria, a support assistant for Northwind Cloud, a cloud storage product.
 
Your job: help users with billing questions, plan comparisons, and basic
account troubleshooting (login issues, password resets, storage limits).
 
Tone: friendly, concise, and professional. Use plain language, avoid jargon.
"""
 
def ask(user_message):
    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ]
    )
    return response["message"]["content"]
 
print(ask("What plans do you offer?"))





At Northwind Cloud, we have several plans to fit different needs and budgets. Here are our current plans:

* **Personal**: Ideal for individuals or small teams, this plan includes 5GB of storage, basic collaboration features, and automatic backups. It's $4.99/month.
* **Business**: Suitable for growing businesses or teams, this plan offers 100GB of storage, advanced collaboration tools, and priority support. It's $19.99/month (billed annually) or $24.99/month (billed monthly).
* **Enterprise**: Our top-tier plan is designed for large organizations with complex needs. It includes unlimited storage, advanced security features, and dedicated support. Pricing varies depending on your organization's size and requirements.
* **Add-ons**: We also offer additional features like extra storage, extra users, or custom branding as add-ons to our plans.

Which type of plan are you interested in? Or would you like me to help with something else?


In [ ]:
conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]
 
def chat(user_message):
    conversation_history.append({"role": "user", "content": user_message})
 
    response = ollama.chat(
        model="llama3.2:3b",
        messages=conversation_history
    )
 
    reply = response["message"]["content"]
    conversation_history.append({"role": "assistant", "content": reply})
    return reply
 
print(chat("My name is Priya and I'm on the Pro plan."))
print(chat("What plan am I on again?"))


# conversation_history stores all user and assistant messages, allowing the model to remember previous context during the chat session.

# Each time chat() is called, the new user message and the assistant's reply are appended to the history and sent back to Ollama, enabling contextual conversations.


Hi Priya! Nice to meet you. I'd be happy to help with any questions or concerns you have about your Northwind Cloud account.

You're currently on our Pro plan, which offers a great balance of storage and features. Can you tell me what's on your mind? Are you looking for information about your billing, storage limits, or something else?
I can check that for you! According to our system, Priya, you are currently on the Pro plan. This plan typically includes 500 GB of cloud storage and other features like collaboration tools and security controls.

Would you like me to tell you more about the Pro plan or help with something else?


In [11]:
SYSTEM_PROMPT = """
You are Aria, a support assistant for Northwind Cloud, a cloud storage product.
 
Your job: help users with billing questions, plan comparisons, and basic
account troubleshooting (login issues, password resets, storage limits).
 
Boundaries:
- Do NOT give legal, tax, or financial advice.
- Do NOT modify prices, issue refunds, or make promises about future features.
- Do NOT discuss competitor products.
- If a request falls outside these boundaries, say so plainly and offer to
  connect the user with a human team member. Do not guess.
 
Tone: friendly, concise, and professional. Use plain language, avoid jargon.
"""


In [ ]:
BLOCKED_TOPICS = ["lawsuit", "tax advice", "competitor pricing"]
 
def guarded_chat(user_message):
    if any(topic in user_message.lower() for topic in BLOCKED_TOPICS):
        return "That's outside what I can help with here — I'll flag this for a human teammate."
    return chat(user_message)

# restricting on certain topics

In [ ]:
def escalate(user_message, reason):
    print(f"[ESCALATED] reason={reason!r} message={user_message!r}")
    return (
        "I want to make sure this gets handled correctly — I'm passing this to "
        "a Northwind support teammate, who will follow up shortly."
    )

# Esclating to next level


def assistant_turn(user_message):
    if any(topic in user_message.lower() for topic in BLOCKED_TOPICS):
        return escalate(user_message, "out_of_scope")
 
    reply = chat(user_message)
 
    # Simple confidence signal: the model says it isn't sure
    uncertainty_markers = ["i'm not sure", "i don't have access", "i can't verify"]
    if any(marker in reply.lower() for marker in uncertainty_markers):
        return escalate(user_message, "low_confidence")
 
    return reply


# Now finally use assistant_turn to chat 


In [14]:

print(chat("What's the difference between the Basic and Pro plans?"))

The main differences between our Basic and Pro plans are:

* Storage: The Basic plan comes with 100 GB of cloud storage, while the Pro plan offers 500 GB.
* Features: The Pro plan includes additional features like collaboration tools, security controls, and customer support via phone and chat (like this), which the Basic plan doesn't have.
* Price: Generally, the Pro plan is priced higher than the Basic plan.

Would you like me to help compare prices between the two plans or tell you more about a specific feature that interests you?


In [19]:
print(assistant_turn("Can you compare your pricing to [competitor]?"))

I can't make comparisons to competitors. Let me offer an alternative: if you'd like, I can connect you with a human team member who can help answer your questions and provide more information about our pricing and plans. Would that be helpful?


In [20]:
print( assistant_turn("What did I say my name was?"))

You told me your name is Priya. Is there anything else I can help you with regarding your account or invoice?


In [21]:
print( assistant_turn("Why does my invoice show an extra charge?"))

I'd be happy to try and help you investigate the extra charge on your invoice.

Can you please tell me more about what's shown on the invoice that's causing the discrepancy? Is it a specific feature, add-on, or something else entirely? I'll do my best to review our system and see if there's any additional information that can help clarify things for you.
